###Initializing Catalog, Schema and Volume

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS quickcart;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.default;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS quickcart.default.source_data;

###Importing required libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

from datetime import datetime, timedelta
import random
import uuid

###Initializing path

In [0]:
PROJECT_NAME = "quickcart"

# Unity Catalog objects
CATALOG = "quickcart"

# Source data location
SOURCE_VOLUME = f"/Volumes/{CATALOG}/default/source_data"

# Individual source locations
CUSTOMER_PATH = f"{SOURCE_VOLUME}/customers"
PRODUCT_PATH = f"{SOURCE_VOLUME}/products"
ORDER_PATH = f"{SOURCE_VOLUME}/orders"
PAYMENT_PATH = f"{SOURCE_VOLUME}/payments"
DELIVERY_PATH = f"{SOURCE_VOLUME}/deliveries"

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"Project       : {PROJECT_NAME}")
print(f"Source Volume : {SOURCE_VOLUME}")

###Customer Data

In [0]:
num_customers = 100_000
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("pincode", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])


# ------------------------------------------------------------
# Reference data
# ------------------------------------------------------------

first_names = [
    "Rahul", "Amit", "Rohan", "Vikas", "Akash",
    "Suresh", "Raj", "Arjun", "Karan", "Vivek",
    "Priya", "Neha", "Sneha", "Pooja", "Anjali",
    "Kavya", "Aarti", "Simran", "Riya", "Nisha"
]

last_names = [
    "Sharma", "Patil", "Kumar", "Singh", "Mehta",
    "Desai", "Joshi", "Gupta", "Verma", "Shah",
    "Pawar", "Kulkarni", "Jadhav", "Mishra", "Reddy"
]

locations = [
    ("Mumbai", "Maharashtra", "400001"),
    ("Pune", "Maharashtra", "411001"),
    ("Nagpur", "Maharashtra", "440001"),
    ("Nashik", "Maharashtra", "422001"),
    ("Delhi", "Delhi", "110001"),
    ("Bangalore", "Karnataka", "560001"),
    ("Hyderabad", "Telangana", "500001"),
    ("Chennai", "Tamil Nadu", "600001"),
    ("Kolkata", "West Bengal", "700001"),
    ("Ahmedabad", "Gujarat", "380001"),
    ("Jaipur", "Rajasthan", "302001"),
    ("Surat", "Gujarat", "395001")
]

segments = [
    "Regular",
    "Premium",
    "VIP"
]

genders = [
    "Male",
    "Female"
]

# ------------------------------------------------------------
# Generate customer records
# ------------------------------------------------------------
customer_data = []

start_date = datetime(2022, 1, 1)
end_date = datetime(2026, 7, 31)

for i in range(1, num_customers+1):
    first_name = random.choice(first_names)
    last_name = random.choice(last_names)

    customer_name = f"{first_name} {last_name}"

    customer_id = f"C{i:06d}"

    email = (
        f"{first_name.lower()}."
        f"{last_name.lower()}"
        f"{i}@quickcart.com"
    )

    phone = f"{random.randint(6000000000, 9999999999)}"

    gender = random.choice(genders)

    # Age between approximately 18 and 65
    dob = datetime(
        random.randint(1960, 2006),
        random.randint(1, 12),
        random.randint(1, 28)
    ).date()

    city, state, pincode = random.choice(locations)

    registration_date = (
        start_date +
        timedelta(
            days=random.randint(
                0,
                (end_date - start_date).days
            )
        )
    ).date()

    segment = random.choices(
        segments,
        weights=[70, 25, 5],
        k=1
    )[0]

    updated_at = datetime(
        2026,
        random.randint(1, 7),
        random.randint(1, 28),
        random.randint(0, 23),
        random.randint(0, 59),
        random.randint(0, 59)
    )

    customer_data.append((
        customer_id,
        customer_name,
        email,
        phone,
        gender,
        dob,
        city,
        state,
        pincode,
        registration_date,
        segment,
        updated_at
    ))

print(f"Generated {len(customer_data):,} customer records")

In [0]:
df_customer = spark.createDataFrame(customer_data, customer_schema)
display(df_customer.limit(5))

#df_customer.groupBy("customer_id").count().filter(f.col("count")>1).show()
'''
df_customer.select([
    F.sum(
        F.when(f.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_customer.columns
]).show()
'''
df_customer.repartition(10).write.mode("overwrite").parquet(CUSTOMER_PATH)

In [0]:
#display(spark.read.parquet(CUSTOMER_PATH).limit(5))

###Products Data

In [0]:
# ============================================================
# PRODUCT DATA GENERATION
# ============================================================

import builtins

NUM_PRODUCTS = 5_000

product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("cost", DoubleType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("product_rating", DoubleType(), True),
    StructField("created_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
])

# ------------------------------------------------------------
# Reference data
# ------------------------------------------------------------

product_categories = {
    "Electronics": [
        "Mobile",
        "Laptop",
        "Tablet",
        "Headphones",
        "Smartwatch",
        "Camera"
    ],
    "Fashion": [
        "Men Clothing",
        "Women Clothing",
        "Shoes",
        "Bags",
        "Accessories"
    ],
    "Home & Kitchen": [
        "Furniture",
        "Kitchen Appliances",
        "Home Decor",
        "Cookware",
        "Storage"
    ],
    "Beauty": [
        "Skincare",
        "Haircare",
        "Makeup",
        "Fragrance"
    ],
    "Sports": [
        "Fitness",
        "Running",
        "Cricket",
        "Football",
        "Outdoor"
    ],
    "Grocery": [
        "Snacks",
        "Beverages",
        "Staples",
        "Dairy",
        "Packaged Food"
    ],
    "Books": [
        "Fiction",
        "Non-Fiction",
        "Technology",
        "Business",
        "Education"
    ],
    "Toys": [
        "Educational",
        "Action Figures",
        "Board Games",
        "Remote Control",
        "Kids Games"
    ]
}

brands = {
    "Electronics": [
        "Apple", "Samsung", "Sony", "OnePlus",
        "Dell", "HP", "Lenovo", "Boat"
    ],
    "Fashion": [
        "Nike", "Adidas", "Puma", "Levis",
        "Allen Solly", "Roadster"
    ],
    "Home & Kitchen": [
        "Philips", "Prestige", "Havells",
        "Ikea", "Bajaj", "Pigeon"
    ],
    "Beauty": [
        "Lakme", "Maybelline", "Loreal",
        "Nivea", "Dove", "Mamaearth"
    ],
    "Sports": [
        "Nike", "Adidas", "Puma",
        "Yonex", "Cosco", "SG"
    ],
    "Grocery": [
        "Nestle", "Britannia", "Tata",
        "Amul", "Parle", "Haldiram"
    ],
    "Books": [
        "Penguin", "HarperCollins",
        "Oxford", "McGraw Hill"
    ],
    "Toys": [
        "Lego", "Mattel", "Funskool",
        "Hasbro", "Hot Wheels"
    ]
}

# ------------------------------------------------------------
# Generate products
# ------------------------------------------------------------

product_data = []

product_start_date = datetime(2023, 1, 1)

for i in range(1, NUM_PRODUCTS + 1):

    product_id = f"P{i:05d}"

    category = random.choice(
        list(product_categories.keys())
    )

    subcategory = random.choice(
        product_categories[category]
    )

    brand = random.choice(
        brands[category]
    )

    product_name = f"{brand} {subcategory} Product {i}"

    # Generate realistic price ranges by category
    if category == "Electronics":
        price = builtins.round(random.uniform(1_000, 150_000), 2)

    elif category == "Fashion":
        price = builtins.round(random.uniform(300, 15_000), 2)

    elif category == "Home & Kitchen":
        price = builtins.round(random.uniform(200, 50_000), 2)

    elif category == "Beauty":
        price = builtins.round(random.uniform(100, 8_000), 2)

    elif category == "Sports":
        price = builtins.round(random.uniform(300, 20_000), 2)

    elif category == "Grocery":
        price = builtins.round(random.uniform(50, 5_000), 2)

    elif category == "Books":
        price = builtins.round(random.uniform(100, 3_000), 2)

    else:
        price = builtins.round(random.uniform(200, 10_000), 2)

    # Product cost between 55% and 85% of selling price
    cost = builtins.round(
        price * random.uniform(0.55, 0.85),
        2
    )

    supplier_id = f"SUP{random.randint(1, 500):04d}"

    product_rating = builtins.round(
        random.uniform(2.5, 5.0),
        1
    )

    created_date = (
        product_start_date +
        timedelta(
            days=random.randint(
                0,
                1_000
            )
        )
    ).date()

    updated_at = datetime(
        2026,
        random.randint(1, 7),
        random.randint(1, 28),
        random.randint(0, 23),
        random.randint(0, 59),
        random.randint(0, 59)
    )

    product_data.append((
        product_id,
        product_name,
        category,
        subcategory,
        brand,
        price,
        cost,
        supplier_id,
        product_rating,
        created_date,
        updated_at
    ))

print(f"Generated {len(product_data):,} products")

In [0]:
df_products = spark.createDataFrame(product_data, product_schema)

df_products.repartition(5).write.mode('overwrite').parquet(PRODUCT_PATH)

###Orders Data

In [0]:
# ============================================================
# ORDER DATA GENERATION
# ============================================================

NUM_ORDERS = 1_000_000

# Read the previously generated source data
customers_source = spark.read.parquet(CUSTOMER_PATH)
products_source = spark.read.parquet(PRODUCT_PATH)
NUM_CUSTOMERS = customers_source.count()
NUM_PRODUCTS = products_source.count()

customer_ids = customers_source.select("customer_id")
product_reference = products_source.select(
    "product_id",
    "price"
)

# ------------------------------------------------------------
# Generate base order IDs
# ------------------------------------------------------------

orders_df = (
    spark.range(1, NUM_ORDERS + 1)
    .withColumn(
        "order_id",
        F.format_string("O%08d", F.col("id"))
    )
    .drop("id")
)

# ------------------------------------------------------------
# Assign customers
# ------------------------------------------------------------

orders_df = (
    orders_df
    .withColumn(
        "customer_number",
        (
            F.floor(
                F.rand(RANDOM_SEED) * NUM_CUSTOMERS
            ) + 1
        ).cast("long")
    )
    .withColumn(
        "customer_id",
        F.format_string(
            "C%06d",
            F.col("customer_number")
        )
    )
    .drop("customer_number")
)

# ------------------------------------------------------------
# Assign products
# ------------------------------------------------------------

orders_df = (
    orders_df
    .withColumn(
        "product_number",
        (
            F.floor(
                F.rand(RANDOM_SEED + 1) * NUM_PRODUCTS
            ) + 1
        ).cast("long")
    )
    .withColumn(
        "product_id",
        F.format_string(
            "P%05d",
            F.col("product_number")
        )
    )
    .drop("product_number")
)

# ------------------------------------------------------------
# Join product price
# ------------------------------------------------------------

orders_df = (
    orders_df
    .join(
        F.broadcast(product_reference),
        on="product_id",
        how="left"
    )
)

# ------------------------------------------------------------
# Quantity
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "quantity",
    (
        F.floor(
            F.rand(RANDOM_SEED + 2) * 5
        ) + 1
    ).cast("int")
)

# ------------------------------------------------------------
# Discount
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "discount_percentage",
    F.round(
        F.rand(RANDOM_SEED + 3) * 30,
        2
    )
)

orders_df = orders_df.withColumn(
    "discount_amount",
    F.round(
        F.col("price")
        * F.col("quantity")
        * F.col("discount_percentage")
        / 100,
        2
    )
)

# ------------------------------------------------------------
# Order amount
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "order_amount",
    F.round(
        (
            F.col("price") * F.col("quantity")
        ) - F.col("discount_amount"),
        2
    )
)

# ------------------------------------------------------------
# Payment method
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "payment_method",
    F.expr("""
        CASE
            WHEN rand() < 0.40 THEN 'UPI'
            WHEN rand() < 0.65 THEN 'Credit Card'
            WHEN rand() < 0.80 THEN 'Debit Card'
            WHEN rand() < 0.90 THEN 'Net Banking'
            WHEN rand() < 0.97 THEN 'Wallet'
            ELSE 'COD'
        END
    """)
)

# ------------------------------------------------------------
# Order status
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "order_status",
    F.expr("""
        CASE
            WHEN rand() < 0.70 THEN 'DELIVERED'
            WHEN rand() < 0.82 THEN 'SHIPPED'
            WHEN rand() < 0.90 THEN 'CONFIRMED'
            WHEN rand() < 0.95 THEN 'PLACED'
            WHEN rand() < 0.98 THEN 'CANCELLED'
            ELSE 'RETURNED'
        END
    """)
)

# ------------------------------------------------------------
# Order timestamp
# July 2025 - July 2026
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "order_timestamp",
    F.expr("""
        timestampadd(
            SECOND,
            cast(rand() * 31536000 as int),
            timestamp('2025-07-01 00:00:00')
        )
    """)
)

# ------------------------------------------------------------
# Shipping city
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "shipping_city",
    F.element_at(
        F.array(
            F.lit("Mumbai"),
            F.lit("Pune"),
            F.lit("Delhi"),
            F.lit("Bangalore"),
            F.lit("Hyderabad"),
            F.lit("Chennai"),
            F.lit("Kolkata"),
            F.lit("Ahmedabad"),
            F.lit("Jaipur"),
            F.lit("Surat"),
            F.lit("Nagpur"),
            F.lit("Nashik")
        ),
        (
            F.floor(
                F.rand(RANDOM_SEED + 4) * 12
            ) + 1
        ).cast("int")
    )
)

# ------------------------------------------------------------
# Updated timestamp
# ------------------------------------------------------------

orders_df = orders_df.withColumn(
    "updated_at",
    F.col("order_timestamp") +
    F.expr("INTERVAL 1 DAY")
)

# Select final columns
orders_df = orders_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "price",
    "discount_percentage",
    "discount_amount",
    "order_amount",
    "payment_method",
    "order_status",
    "order_timestamp",
    "shipping_city",
    "updated_at"
)

orders_df.printSchema()

In [0]:
#display(orders_df.limit(20))
orders_df.repartition(20).write.mode("overwrite").parquet(ORDER_PATH)


###Payment Data

In [0]:
# ============================================================
# PAYMENT DATA GENERATION
# ============================================================

orders_for_payment = (
    spark.read
    .parquet(ORDER_PATH)
    .select(
        "order_id",
        "order_amount",
        "payment_method",
        "order_timestamp",
        "order_status"
    )
)

payments_df = (
    orders_for_payment
    .withColumn(
        "payment_id",
        F.concat(
            F.lit("PAY"),
            F.substring("order_id", 2, 8)
        )
    )
    .withColumn(
        "transaction_amount",
        F.col("order_amount")
    )
    .withColumn(
        "payment_status",
        F.when(
            F.col("order_status") == "CANCELLED",
            F.lit("FAILED")
        )
        .when(
            F.col("order_status") == "RETURNED",
            F.lit("REFUNDED")
        )
        .when(
            F.rand(RANDOM_SEED + 10) < 0.95,
            F.lit("SUCCESS")
        )
        .when(
            F.rand(RANDOM_SEED + 11) < 0.50,
            F.lit("PENDING")
        )
        .otherwise(
            F.lit("FAILED")
        )
    )
    .withColumn(
        "transaction_timestamp",
        F.col("order_timestamp") +
        F.expr(
            "INTERVAL 1 HOUR"
        )
    )
    .withColumn(
        "transaction_reference",
        F.concat(
            F.lit("TXN-"),
            F.upper(
                F.substring(
                    F.sha2(
                        F.col("order_id"),
                        256
                    ),
                    1,
                    12
                )
            )
        )
    )
    .select(
        "payment_id",
        "order_id",
        "payment_method",
        "payment_status",
        "transaction_amount",
        "transaction_timestamp",
        "transaction_reference"
    )
)

display(payments_df.limit(20))

In [0]:
payments_df.repartition(20).write.mode("overwrite").parquet(PAYMENT_PATH)

###Delivery Data

In [0]:
# ============================================================
# DELIVERY DATA GENERATION
# ============================================================

orders_for_delivery = (
    spark.read
    .parquet(ORDER_PATH)
    .filter(
        F.col("order_status").isin(
            "CONFIRMED",
            "SHIPPED",
            "DELIVERED",
            "RETURNED"
        )
    )
    .select(
        "order_id",
        "shipping_city",
        "order_timestamp",
        "order_status"
    )
)

deliveries_df = (
    orders_for_delivery
    .withColumn(
        "delivery_id",
        F.concat(
            F.lit("DEL"),
            F.substring("order_id", 2, 8)
        )
    )
    .withColumn(
        "delivery_partner",
        F.element_at(
            F.array(
                F.lit("QuickShip"),
                F.lit("FastTrack"),
                F.lit("BlueDart"),
                F.lit("Delhivery"),
                F.lit("EcomExpress")
            ),
            (
                F.floor(
                    F.rand(RANDOM_SEED + 20) * 5
                ) + 1
            ).cast("int")
        )
    )
    .withColumn(
        "warehouse",
        F.element_at(
            F.array(
                F.lit("WH-MUM"),
                F.lit("WH-PUN"),
                F.lit("WH-DEL"),
                F.lit("WH-BLR"),
                F.lit("WH-HYD")
            ),
            (
                F.floor(
                    F.rand(RANDOM_SEED + 21) * 5
                ) + 1
            ).cast("int")
        )
    )
)

In [0]:
# ============================================================
# DELIVERY DATES
# ============================================================

deliveries_df = (
    deliveries_df
    .withColumn(
        "shipped_date",
        F.col("order_timestamp") +
        F.expr(
            "INTERVAL 1 DAY"
        )
    )
    .withColumn(
        "estimated_delivery_date",
        F.col("order_timestamp") +
        F.expr(
            "INTERVAL 5 DAYS"
        )
    )
)

In [0]:
deliveries_df = (
    deliveries_df
    .withColumn(
        "actual_delivery_date",
        F.when(
            F.col("order_status") == "DELIVERED",
            F.col("order_timestamp") + F.expr("INTERVAL 1 DAY") * (2 + (F.rand(RANDOM_SEED + 25) * 6).cast("int"))
        )
    )
)

In [0]:
deliveries_df = (
    deliveries_df
    .withColumn(
        "delivery_status",
        F.when(
            F.col("order_status") == "DELIVERED",
            F.lit("DELIVERED")
        )
        .when(
            F.col("order_status") == "RETURNED",
            F.lit("RETURNED")
        )
        .when(
            F.col("order_status") == "SHIPPED",
            F.lit("IN_TRANSIT")
        )
        .otherwise(
            F.lit("PROCESSING")
        )
    )
)

In [0]:
deliveries_df = (
    deliveries_df
    .withColumn(
        "delivery_attempts",
        F.when(
            F.col("delivery_status") == "DELIVERED",
            F.floor(
                F.rand(RANDOM_SEED + 22) * 3
            ).cast("int") + 1
        )
        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
deliveries_df = deliveries_df.select(
    "delivery_id",
    "order_id",
    "delivery_partner",
    "warehouse",
    "shipping_city",
    "delivery_status",
    "order_timestamp",
    "shipped_date",
    "estimated_delivery_date",
    "actual_delivery_date",
    "delivery_attempts"
)

display(deliveries_df.limit(20))

In [0]:
deliveries_df.repartition(15).write.mode("overwrite").parquet(DELIVERY_PATH)

###Validation

In [0]:
customers = spark.read.parquet(CUSTOMER_PATH)
products = spark.read.parquet(PRODUCT_PATH)
orders = spark.read.parquet(ORDER_PATH)
payments = spark.read.parquet(PAYMENT_PATH)
deliveries = spark.read.parquet(DELIVERY_PATH)

print("Customers: ",customers.count())
print("Products: ",products.count())
print("Orders: ",orders.count())
print("Payments: ",payments.count())
print("Deliveries: ",deliveries.count())

###Problematic Data

In [0]:
# ============================================================
# SOURCE-SYSTEM DATA ISSUES
# ============================================================

ISSUES_PATH = f"{SOURCE_VOLUME}/issues"

CUSTOMER_ISSUES_PATH = f"{ISSUES_PATH}/customers"
ORDER_ISSUES_PATH = f"{ISSUES_PATH}/orders"
PAYMENT_ISSUES_PATH = f"{ISSUES_PATH}/payments"
DELIVERY_ISSUES_PATH = f"{ISSUES_PATH}/deliveries"

print("Issues path:", ISSUES_PATH)

In [0]:
# ============================================================
# ISSUE 1: DUPLICATE / UPDATED CUSTOMER RECORDS
# ============================================================

customers = spark.read.parquet(CUSTOMER_PATH)

# Select 5% of customers for updates
customer_updates = (
    customers
    .sample(
        withReplacement=False,
        fraction=0.05,
        seed=100
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Mumbai"),
                F.lit("Pune"),
                F.lit("Bangalore"),
                F.lit("Delhi"),
                F.lit("Hyderabad")
            ),
            (
                F.floor(
                    F.rand(101) * 5
                ) + 1
            ).cast("int")
        )
    )
    .withColumn(
        "state",
        F.when(
            F.col("city") == "Mumbai",
            "Maharashtra"
        )
        .when(
            F.col("city") == "Pune",
            "Maharashtra"
        )
        .when(
            F.col("city") == "Bangalore",
            "Karnataka"
        )
        .when(
            F.col("city") == "Delhi",
            "Delhi"
        )
        .otherwise("Telangana")
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

print(
    "Customer update records:",
    customer_updates.count()
)

display(customer_updates.limit(10))

In [0]:
customer_updates.repartition(2).write.mode("overwrite").parquet(f"{CUSTOMER_ISSUES_PATH}/updates")

In [0]:
# ============================================================
# ISSUE 2: DUPLICATE / UPDATED ORDERS
# ============================================================

orders = spark.read.parquet(ORDER_PATH)

order_updates = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.02,
        seed=200
    )
    .withColumn(
        "order_status",
        F.when(
            F.col("order_status") == "PLACED",
            "CONFIRMED"
        )
        .when(
            F.col("order_status") == "CONFIRMED",
            "SHIPPED"
        )
        .when(
            F.col("order_status") == "SHIPPED",
            "DELIVERED"
        )
        .otherwise(
            F.col("order_status")
        )
    )
    .withColumn(
        "updated_at",
        F.col("updated_at") +
        F.expr("INTERVAL 2 HOURS")
    )
)

print(
    "Order update records:",
    order_updates.count()
)

display(order_updates.limit(10))

order_updates.repartition(5).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/updates")

In [0]:
# ============================================================
# ISSUE 3: NULL VALUES
# ============================================================

customer_nulls = (
    customers
    .sample(
        withReplacement=False,
        fraction=0.03,
        seed=300
    )
    .withColumn(
        "phone",
        F.lit(None).cast("string")
    )
    .withColumn(
        "email",
        F.lit(None).cast("string")
    )
)

print(
    "Customers with NULL attributes:",
    customer_nulls.count()
)

display(customer_nulls.limit(10))

customer_nulls.repartition(2).write.mode("overwrite").parquet(f"{CUSTOMER_ISSUES_PATH}/null_records")

In [0]:
# ============================================================
# ISSUE 4A: INVALID CUSTOMER REFERENCES
# ============================================================

invalid_customer_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=400
    )
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("INVALID_C"),
            F.substring(
                F.col("order_id"),
                2,
                8
            )
        )
    )
)

print(
    "Orders with invalid customers:",
    invalid_customer_orders.count()
)

display(
    invalid_customer_orders.limit(10)
)

invalid_customer_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/invalid_customers")

In [0]:
# ============================================================
# ISSUE 4B: INVALID PRODUCT REFERENCES
# ============================================================

invalid_product_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=500
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("INVALID_P"),
            F.substring(
                F.col("order_id"),
                2,
                8
            )
        )
    )
)

print(
    "Orders with invalid products:",
    invalid_product_orders.count()
)

display(
    invalid_product_orders.limit(10)
)

invalid_product_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/invalid_products")

In [0]:
# ============================================================
# ISSUE 5: INVALID BUSINESS VALUES
# ============================================================

invalid_business_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=600
    )
    .withColumn(
        "quantity",
        F.when(
            F.rand(601) < 0.33,
            F.lit(-1)
        )
        .when(
            F.rand(602) < 0.66,
            F.lit(0)
        )
        .otherwise(
            F.lit(1)
        )
    )
    .withColumn(
        "discount_percentage",
        F.when(
            F.rand(603) < 0.5,
            F.lit(150.0)
        )
        .otherwise(
            F.lit(-20.0)
        )
    )
    .withColumn(
        "order_amount",
        F.when(
            F.rand(604) < 0.5,
            F.lit(-500.0)
        )
        .otherwise(
            F.lit(0.0)
        )
    )
)

display(
    invalid_business_orders.limit(10)
)

invalid_business_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/invalid_business_values")

In [0]:
# ============================================================
# ISSUE 6: LATE ARRIVING ORDERS
# ============================================================

late_orders = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=700
    )
    .withColumn(
        "order_timestamp",
        F.to_timestamp(
            F.lit("2026-07-10 10:00:00")
        )
    )
    .withColumn(
        "updated_at",
        F.to_timestamp(
            F.lit("2026-07-11 15:00:00")
        )
    )
)

display(
    late_orders.limit(10)
)

late_orders.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/late_arrivals")

In [0]:
# ============================================================
# ISSUE 7: NULL TRANSACTION ATTRIBUTES
# ============================================================

order_nulls = (
    orders
    .sample(
        withReplacement=False,
        fraction=0.005,
        seed=800
    )
    .withColumn(
        "customer_id",
        F.lit(None).cast("string")
    )
    .withColumn(
        "payment_method",
        F.lit(None).cast("string")
    )
    .withColumn(
        "shipping_city",
        F.lit(None).cast("string")
    )
)

display(
    order_nulls.limit(10)
)

order_nulls.repartition(2).write.mode("overwrite").parquet(f"{ORDER_ISSUES_PATH}/null_records")

In [0]:
# ============================================================
# SOURCE ISSUE SUMMARY
# ============================================================

issue_summary = [
    ("Duplicate/Updated Customers", customer_updates.count()),
    ("Customer NULL Records", customer_nulls.count()),
    ("Duplicate/Updated Orders", order_updates.count()),
    ("Invalid Customer References", invalid_customer_orders.count()),
    ("Invalid Product References", invalid_product_orders.count()),
    ("Invalid Business Values", invalid_business_orders.count()),
    ("Late Arriving Orders", late_orders.count()),
    ("Order NULL Records", order_nulls.count())
]

issue_summary_df = spark.createDataFrame(
    issue_summary,
    ["issue_type", "record_count"]
)

display(issue_summary_df)